# Spacecraft - PPO

**VecNormalize** scales the observations and rewards using running statistics, which helps with your large goal bonus.
It keeps those statistics in the vec env, so they are saved to a `.pkl` file next to the model and must be loaded again for evaluation.


In [1]:
pip install mujoco gymnasium stable-baselines3[extra] sympy psutil

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


**Optional:** check how many CPU cores you can use

In [2]:
import os
import psutil

print(f"Logical cores (threads): {os.cpu_count()}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")

Logical cores (threads): 16
Physical cores: 8


In [3]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import Spacecraft_V1

**Register the env**

In [4]:
gym.register(
    id="Spacecraft_V1",
    entry_point="Spacecraft_V1:SpacecraftEnv",
    max_episode_steps=2000,
)

**Declare the vectorized env (with VecNormalize) and the model**

In [8]:
# 4 copies of the env, all running in this process
vec_env = make_vec_env("Spacecraft_V1", n_envs=6, vec_env_cls=DummyVecEnv)

# normalize observations and rewards; clip_obs keeps the normalized observations in a sane range
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

model = PPO("MlpPolicy", vec_env, verbose=1, device="cpu")

Using cpu device


**Train**

In [9]:
model.learn(total_timesteps=500000)

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 531       |
|    ep_rew_mean     | -1.15e+03 |
| time/              |           |
|    fps             | 4476      |
|    iterations      | 1         |
|    time_elapsed    | 2         |
|    total_timesteps | 12288     |
----------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 532         |
|    ep_rew_mean          | -1.37e+03   |
| time/                   |             |
|    fps                  | 2296        |
|    iterations           | 2           |
|    time_elapsed         | 10          |
|    total_timesteps      | 24576       |
| train/                  |             |
|    approx_kl            | 0.010086745 |
|    clip_fraction        | 0.127       |
|    clip_range           | 0.2         |
|    entropy_loss         | -12.8       |
|    explained_variance   | 0.383       |
|    learning_rate  

**Save the model and the normalization statistics**

In [10]:
model.save("ppo_norm_spacecraft")
vec_env.save("ppo_norm_stats.pkl")   # the normalization statistics, needed to evaluate the model
vec_env.close()

**Evaluate model**

In [12]:
model = PPO.load("ppo_norm_spacecraft")
# the model trained on normalized observations, so load the saved statistics to normalize them the same way
stats = VecNormalize.load("ppo_norm_stats.pkl", DummyVecEnv([lambda: gym.make("Spacecraft_V1")]))

eval_env = gym.make("Spacecraft_V1", render_mode="human")
obs, info = eval_env.reset()
for _ in range(2000):
    action, _states = model.predict(stats.normalize_obs(obs), deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    if terminated or truncated:
        obs, info = eval_env.reset()
eval_env.close()